# Untappd Business API Data Extraction

This notebook demonstrates how to extract and analyze data from your Untappd Business account using the API.

## What we'll cover:
1. Setting up API authentication
2. Fetching checkins data from your location
3. Extracting beer information
4. Basic data analysis and visualization
5. Exporting data for your menu

## Prerequisites:
- Untappd Business account
- API Read-Only token
- Location ID for your venue

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
from collections import Counter

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 2. Set Up API Credentials

Configure your Untappd Business API credentials here. You can find these in your [Untappd Business account](https://business.untappd.com/account).

In [ ]:
# 🔑 API Configuration
# Update these values with your actual credentials

CONFIG = {
    'email': 'mestredacervejapt@gmail.com',  # Your Untappd account email
    'token': '-cksQmzXz8cxgUfpAf_C',     # Your Read-Only token (already provided)
    'location_id': None,  # Will be set after running the location finder below
}

# API endpoints
BASE_URL = 'https://business.untappd.com/api/v1'

# Verify configuration
if CONFIG['email'] == 'your-email@untappd.com':
    print("⚠️  Please update the CONFIG with your actual email!")
elif CONFIG['location_id'] is None:
    print("✅ Email configured!")
    print("ℹ️  Run the 'Find Your Location ID' section below to get your location_id")
else:
    print("✅ Configuration complete!")
    print(f"Email: {CONFIG['email']}")
    print(f"Location ID: {CONFIG['location_id']}")

## 3. API Functions

Let's create helper functions for API authentication and data fetching.

In [ ]:
def create_auth_header(email, token):
    """Create Basic Auth header for Untappd Business API"""
    import base64
    credentials = f"{email}:{token}"
    encoded_credentials = base64.b64encode(credentials.encode('utf-8')).decode('utf-8')
    return {'Authorization': f'Basic {encoded_credentials}'}

def make_api_request(endpoint, params=None, max_retries=3):
    """Make authenticated API request with error handling"""
    headers = create_auth_header(CONFIG['email'], CONFIG['token'])
    url = f"{BASE_URL}{endpoint}"

    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, params=params, timeout=30)

            if response.status_code == 200:
                return response.json()
            elif response.status_code == 401:
                print("❌ Authentication failed. Check your credentials.")
                return None
            elif response.status_code == 403:
                print("❌ Access forbidden. Check your token permissions.")
                return None
            elif response.status_code == 429:
                print(f"⚠️  Rate limited. Waiting before retry {attempt + 1}/{max_retries}...")
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            else:
                print(f"❌ API Error {response.status_code}: {response.text}")
                return None

        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
            return None

    print("❌ All retry attempts failed")
    return None

def get_location_checkins(location_id, checkin_type='on_premise', limit=50):
    """Fetch checkins for a specific location"""
    endpoint = f'/locations/{location_id}/checkins'
    params = {
        'type': checkin_type,
        'limit': limit
    }

    data = make_api_request(endpoint, params)
    if data and 'checkins' in data:
        return data['checkins']
    else:
        print("❌ Failed to fetch checkins")
        return []

def get_location_info(location_id):
    """Get basic information about a location"""
    endpoint = f'/locations/{location_id}'
    return make_api_request(endpoint)

print("API functions defined successfully!")

## 3.5. Find Your Location ID

If you don't know your location ID, let's fetch your available locations from Untappd Business.

In [ ]:
def get_locations():
    """Fetch all locations associated with your Untappd Business account"""
    endpoint = '/locations'
    return make_api_request(endpoint)

# Fetch your locations
print("🔍 Fetching your Untappd Business locations...")
locations_data = get_locations()

if locations_data and 'locations' in locations_data:
    locations = locations_data['locations']
    print(f"✅ Found {len(locations)} location(s)")

    # Display locations in a nice format
    print("\n📍 Your Locations:")
    print("-" * 60)

    for i, location in enumerate(locations, 1):
        loc_id = location.get('id', 'N/A')
        name = location.get('name', 'Unknown')
        address = location.get('address', 'No address')
        city = location.get('city', 'Unknown city')
        state = location.get('state', 'Unknown state')
        country = location.get('country', 'Unknown country')

        print(f"{i}. Location ID: {loc_id}")
        print(f"   Name: {name}")
        print(f"   Address: {address}, {city}, {state}, {country}")
        print()

    # Ask user to identify their location
    print("🎯 Which location is your MESTRE bar?")
    print("Look for the location that matches your bar's address.")
    print("Copy the Location ID from the list above and update CONFIG['location_id']")

else:
    print("❌ Could not fetch locations. Please check:")
    print("1. Your email and token are correct")
    print("2. You have access to Untappd Business")
    print("3. Your account has at least one location set up")
    print("\nAlternatively, visit: https://business.untappd.com/locations")
    print("And look for your MESTRE location - the ID will be in the URL or interface")

### ⚠️ No Locations Found?

If the API returns 0 locations, you need to set up your venue in Untappd Business first:

1. **Go to [business.untappd.com](https://business.untappd.com)**
2. **Sign in** with your Untappd account
3. **Click "Locations"** in the left menu
4. **Click "Add Location"** or "Create Location"
5. **Fill in your MESTRE bar details**:
   - Name: MESTRE
   - Address: Your bar's full address
   - Phone: Contact number
   - Website: (optional)
6. **Save the location**
7. **Come back to this notebook** and re-run the location finder

**Alternative**: You can also find the Location ID manually:
- Visit: https://business.untappd.com/locations
- Look at the URL when viewing your location: `.../locations/[LOCATION_ID]`
- Or check the location details in the interface

Once you have the Location ID, update the `YOUR_CHOSEN_LOCATION_ID` variable below.

### 🎯 Great! You Found Your Venue

You found your venue at: https://untappd.com/v/mestre-da-cerveja/11094009

**However**: This is your *regular Untappd venue ID* (11094009). For the Business API, you need to *claim this venue* in Untappd Business first.

### 📋 What to Do Next:

1. **Go to [business.untappd.com](https://business.untappd.com)**
2. **Sign in** with your Untappd account
3. **Click "Locations"** → **"Add Location"**
4. **Search for "MESTRE"** or paste your venue URL
5. **Claim your venue** and save it
6. **Copy the Business Location ID** (different from 11094009)

The Business Location ID will appear in your Business dashboard after claiming.

### 🔍 Manual Location ID Lookup

If the API method doesn't work, here's how to find your Location ID manually:

1. **Visit your Untappd Business dashboard**: https://business.untappd.com
2. **Go to Locations**: Click "Locations" in the left sidebar
3. **Find your MESTRE location** in the list
4. **Check the URL**: When you view the location details, the URL will be something like:
   ```
   https://business.untappd.com/locations/12345
   ```
   The number `12345` is your Location ID.

5. **Alternative**: Some interfaces show the Location ID directly in the location details.

**Once you have the ID, enter it in the cell below and re-run the notebook.**

### 🧪 Quick Test Script

For easier testing, I've created a simple script `test_api.py` in your folder. You can run it from the terminal:

```bash
python test_api.py
```

**Before running**: Edit `test_api.py` and set your `LOCATION_ID` at the top of the file.

This will quickly verify your API credentials work and show you sample beer data.

## 3.6. Set Your Location ID

After running the location finder above, update your CONFIG with the correct location ID.

In [ ]:
# 🔧 Set Your Location ID
# Replace 'YOUR_LOCATION_ID_HERE' with the actual ID from the list above

YOUR_CHOSEN_LOCATION_ID = 'YOUR_LOCATION_ID_HERE'  # <-- Change this!

# Update CONFIG
if YOUR_CHOSEN_LOCATION_ID != 'YOUR_LOCATION_ID_HERE':
    CONFIG['location_id'] = YOUR_CHOSEN_LOCATION_ID
    print(f"✅ Location ID set to: {CONFIG['location_id']}")
    print("🎉 You're ready to fetch beer data!")
else:
    print("⚠️  Please set YOUR_CHOSEN_LOCATION_ID above with your actual location ID")
    print("Then re-run this cell")

# Verify final configuration
print(f"\n📋 Current CONFIG:")
print(f"Email: {CONFIG['email']}")
print(f"Token: {CONFIG['token'][:10]}...")  # Show only first 10 chars for security
print(f"Location ID: {CONFIG['location_id']}")

if CONFIG['location_id'] and CONFIG['location_id'] != 'YOUR_LOCATION_ID_HERE':
    print("✅ All set! You can now run the data fetching sections.")
else:
    print("❌ Please set your location ID above.")

## 4. Fetch Checkins Data

Let's fetch the recent checkins from your location and extract beer information.

In [ ]:
# Fetch checkins data
print("🔍 Fetching checkins data...")
checkins = get_location_checkins(CONFIG['location_id'], limit=100)

if checkins:
    print(f"✅ Successfully fetched {len(checkins)} checkins")
else:
    print("❌ No checkins data received. Please check your configuration.")

# Let's examine the structure of the first checkin
if checkins:
    print("\n📋 Sample checkin structure:")
    print(json.dumps(checkins[0], indent=2, default=str)[:1000] + "...")

## 5. Process and Analyze Data

Extract beer information from checkins and create a clean dataset.

In [ ]:
def extract_beer_data(checkins):
    """Extract unique beer data from checkins"""
    beers = []

    for checkin in checkins:
        beer = checkin.get('beer', {})
        brewery = beer.get('brewery', {})

        beer_info = {
            'beer_id': beer.get('id'),
            'beer_name': beer.get('name', 'Unknown'),
            'brewery_name': brewery.get('name', 'Unknown Brewery'),
            'style': beer.get('style', 'Unknown Style'),
            'abv': beer.get('abv', 0),
            'ibu': beer.get('ibu', 0),
            'rating': checkin.get('rating_score', 0),
            'checkin_count': beer.get('stats', {}).get('total_count', 0),
            'last_checkin': checkin.get('created_at'),
            'user_name': checkin.get('user', {}).get('user_name', 'Unknown')
        }
        beers.append(beer_info)

    return beers

def get_unique_beers(beers_data):
    """Get unique beers (remove duplicates)"""
    seen_beers = set()
    unique_beers = []

    for beer in beers_data:
        beer_key = (beer['beer_id'], beer['beer_name'])
        if beer_key not in seen_beers:
            seen_beers.add(beer_key)
            unique_beers.append(beer)

    return unique_beers

# Process the data
if checkins:
    all_beers = extract_beer_data(checkins)
    unique_beers = get_unique_beers(all_beers)

    print(f"📊 Total checkins: {len(checkins)}")
    print(f"🍺 Unique beers: {len(unique_beers)}")

    # Create DataFrame
    df_beers = pd.DataFrame(unique_beers)
    df_checkins = pd.DataFrame(all_beers)

    print(f"\n📋 Beer data columns: {list(df_beers.columns)}")
    print(f"\n🍻 Top 5 beers by checkin count:")
    if not df_beers.empty:
        top_beers = df_beers.nlargest(5, 'checkin_count')[['beer_name', 'brewery_name', 'checkin_count', 'rating']]
        print(top_beers.to_string(index=False))
else:
    print("❌ No data to process. Please check your API configuration.")
    df_beers = pd.DataFrame()
    df_checkins = pd.DataFrame()

## 6. Data Visualization

Let's create some visualizations to understand your beer data better.

In [ ]:
if not df_beers.empty:
    # Set up the plotting area
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('MESTRE Beer Analytics Dashboard', fontsize=16, fontweight='bold')

    # 1. Top 10 beers by checkin count
    top_10_beers = df_beers.nlargest(10, 'checkin_count')
    axes[0, 0].barh(top_10_beers['beer_name'].str[:30], top_10_beers['checkin_count'])
    axes[0, 0].set_title('Top 10 Beers by Check-in Count')
    axes[0, 0].set_xlabel('Check-ins')

    # 2. Beer styles distribution
    style_counts = df_beers['style'].value_counts().head(10)
    axes[0, 1].pie(style_counts.values, labels=style_counts.index, autopct='%1.1f%%')
    axes[0, 1].set_title('Beer Styles Distribution')

    # 3. ABV distribution
    df_beers_clean = df_beers[df_beers['abv'] > 0]  # Remove beers with no ABV data
    if not df_beers_clean.empty:
        axes[1, 0].hist(df_beers_clean['abv'], bins=20, edgecolor='black')
        axes[1, 0].set_title('ABV Distribution')
        axes[1, 0].set_xlabel('ABV (%)')
        axes[1, 0].set_ylabel('Number of Beers')
        axes[1, 0].axvline(df_beers_clean['abv'].mean(), color='red', linestyle='--',
                          label=f'Mean: {df_beers_clean["abv"].mean():.1f}%')
        axes[1, 0].legend()

    # 4. Brewery distribution
    brewery_counts = df_beers['brewery_name'].value_counts().head(10)
    axes[1, 1].barh(brewery_counts.index.str[:25], brewery_counts.values)
    axes[1, 1].set_title('Top 10 Breweries')
    axes[1, 1].set_xlabel('Number of Beers')

    plt.tight_layout()
    plt.show()

    # Additional statistics
    print("📈 Key Statistics:")
    print(f"• Total unique beers: {len(df_beers)}")
    print(f"• Average ABV: {df_beers_clean['abv'].mean():.1f}%" if not df_beers_clean.empty else "• ABV data not available")
    print(f"• Most popular style: {df_beers['style'].mode().iloc[0] if not df_beers['style'].mode().empty else 'N/A'}")
    print(f"• Total check-ins analyzed: {len(df_checkins)}")

else:
    print("❌ No data available for visualization. Please check your API configuration.")

## 7. Export Data

Save your beer data for use in your menu or further analysis.

In [ ]:
# Export data to CSV files
if not df_beers.empty:
    # Export unique beers
    df_beers.to_csv('mestre_beers.csv', index=False)
    print("✅ Exported beer data to 'mestre_beers.csv'")

    # Export all checkins
    df_checkins.to_csv('mestre_checkins.csv', index=False)
    print("✅ Exported checkins data to 'mestre_checkins.csv'")

    # Create a simplified version for the menu
    menu_beers = df_beers[['beer_name', 'brewery_name', 'style', 'abv', 'ibu']].copy()
    menu_beers['abv'] = menu_beers['abv'].apply(lambda x: f"{x:.1f}%" if x > 0 else "N/A")
    menu_beers['ibu'] = menu_beers['ibu'].apply(lambda x: f"{int(x)}" if x > 0 else "N/A")
    menu_beers.to_csv('menu_beers.csv', index=False)
    print("✅ Exported simplified menu data to 'menu_beers.csv'")

    # Show sample of menu data
    print("\n📋 Sample menu data:")
    print(menu_beers.head().to_string(index=False))

else:
    print("❌ No data to export. Please check your API configuration.")

print("\n🎯 Next Steps:")
print("1. Use 'menu_beers.csv' in your digital menu")
print("2. Analyze trends in your beer preferences")
print("3. Update your menu when new beers are added")
print("4. Share insights with your brewery partners")

## 🎉 Summary

You've successfully created a data extraction and analysis pipeline for your Untappd Business account! This notebook demonstrates:

- ✅ API authentication with Untappd Business
- ✅ Fetching checkins data from your location
- ✅ Processing and cleaning beer data
- ✅ Data visualization and analytics
- ✅ Exporting data for your digital menu

### Key Features:
- **Error Handling**: Robust API error handling and retries
- **Data Processing**: Extract unique beers from checkins
- **Visualization**: Charts showing beer popularity, styles, and ABV distribution
- **Export**: Multiple CSV formats for different use cases

### For Your Menu:
The exported `menu_beers.csv` file contains the beer data in a format ready for your digital menu. You can use this data to automatically populate your "On Tap" section.

### Rate Limits:
Untappd Business API has rate limits, so this notebook includes proper error handling and respects API constraints.

Happy brewing! 🍺📊